# Mycetoma AI - Explainable Visualizations

This notebook demonstrates how to load a trained model checkpoint and generate **Grad-CAM** explanations for a specific microscopic image. Clinicians can use this to visually verify that the model is making detection based on fungal/bacterial structure grains, rather than background slide artifacts.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import cv2
import matplotlib.pyplot as plt
from src.models.backbone import ResNet50CBAM
from src.models.multi_task_head import MultiTaskHeads
from src.evaluation.xai import CAMExplainer

In [ ]:
# Setup the composite wrapper required for single-task GradCAM gradients
class GradCAMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # Extract just multi-task classification output for CAM
        out = self.model.heads(self.model.backbone(x))
        return out['classification']

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class FullModel(torch.nn.Module):
    def __init__(self):
         super().__init__()
         self.backbone = ResNet50CBAM(pretrained=False)
         self.heads = MultiTaskHeads(in_features=2048, num_classes=3, num_subtypes=10)
         
base_model = FullModel().to(device)

# Optional: load weights here
# base_model.load_state_dict(torch.load('../checkpoints/multitask/best_multi_task_model.pth'))

wrapped_model = GradCAMWrapper(base_model)
wrapped_model.eval()

In [ ]:
# Initialize Explainer with target layer pointing to the last CBAM block output
target_layers = [wrapped_model.model.backbone.layer4[-1]]
explainer = CAMExplainer(wrapped_model, target_layers, use_cuda=torch.cuda.is_available(), cam_type='gradcam')

### Generate Visualizations

Supply an actual image path to run the visualization logic.

In [ ]:
# image_path = "path/to/mycetoma/image.jpg"
# img = cv2.imread(image_path)
# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# rgb_img = np.float32(img) / 255

# Preprocess for CNN
# input_tensor = preprocess(img).unsqueeze(0).to(device)

# visualization, cam_mask = explainer.generate_heatmap(input_tensor, rgb_img, target_class=None)

# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.title("Original")
# plt.imshow(rgb_img)
# plt.subplot(1, 2, 2)
# plt.title("Grad-CAM Heatmap")
# plt.imshow(visualization)
# plt.show()